# Milestone 2 — Encoder trigger head (DeBERTa-v3-large)

First vertical slice of the Path B encoder parser: fine-tune
`microsoft/deberta-v3-large` as a **trigger-identification token classifier**,
then score it with the same word-level metric as the baseline.

**Baseline to beat (trigger id):** F1 `0.735`, and `196.6 ms/sample` — the
encoder does one forward pass per sentence, so watch the ms/sentence drop hard.

Same robustness rules as the reproduction notebook: **clone, don't pip-build**
our code; leave Colab's torch/protobuf/numpy alone; set the protobuf env var
before any import.

> `Runtime → GPU` (A100 recommended), then `Run all`. Mount Drive (Cell 2) if
> you want the trained model to survive a disconnect.

In [ ]:
!nvidia-smi

## 1. Source + deps

Clones the vendored parser (the FrameNet data loaders live there) and installs
the pinned training env. The `encoder_parser/` files themselves are added in the
next cell.

In [ ]:
# The vendored parser (data loaders live here).
![ -d frame-semantic-transformer ] || git clone -q https://github.com/texturejc/frame-semantic-transformer.git
!cd frame-semantic-transformer && git checkout -q 18cb3023bbb6df0c1b53a52182135c0c0132c073

# Install the pinned training env (torch/protobuf/numpy left as Colab ships them).
!pip install -q "transformers>=4.38,<4.45" "tokenizers>=0.15,<0.20" "sentencepiece>=0.1.99" \
  "nltk>=3.8" "accelerate>=0.28,<0.34" "datasets>=2.18,<3.0" "seqeval>=1.2.2"

## 2. Upload the `encoder_parser/` files

Until the project repo is on GitHub, upload `data.py`, `train_trigger.py`,
`eval_trigger.py` next to this notebook. The cell below expects them in the
working dir and expects `frame-semantic-transformer/` one level up from them —
so place `encoder_parser/` as a sibling of `frame-semantic-transformer/`.

Simplest layout on Colab:
```
/content/frame-semantic-transformer/     (cloned above)
/content/encoder_parser/                 (upload data.py, train_trigger.py, eval_trigger.py here)
```
The `data.py` path logic looks for the parser at `../frame-semantic-transformer`.

In [ ]:
import os, sys
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"  # before transformers
sys.path.insert(0, "/content/encoder_parser")

import nltk
for pkg in ["framenet_v17", "wordnet", "omw-1.4"]:
    nltk.download(pkg)

## 3. Sanity-check the data pipeline (cheap, catches alignment bugs early)

Before a multi-hour train, confirm the trigger labels line up with the text.

In [ ]:
from transformers import AutoTokenizer
import data

tok = AutoTokenizer.from_pretrained("microsoft/deberta-v3-large")
print("fast tokenizer:", tok.is_fast)

raw = data.load_trigger_sentences("dev")
print(f"dev sentences: {len(raw)}")
text, locs = raw[0]
print("text :", text)
print("gold trigger words:", [text[s:e] for s, e in data.whitespace_words(text)
                              if s in {data.snap_to_word_start(text, l) for l in locs}])

# Inspect one tokenized example's labels.
ds = data.build_trigger_dataset("dev", tok, max_length=320)
ex = ds[0]
toks = tok.convert_ids_to_tokens(ex["input_ids"])
print("\ntoken -> label (only non -100 shown):")
for t, l in zip(toks, ex["labels"]):
    if l != -100:
        print(f"  {t!r:20} {data.ID2LABEL[l]}")

## 4. Train

~5 epochs over the FrameNet training docs. On an A100 expect roughly 15–40 min.

In [ ]:
import train_trigger

OUTPUT_DIR = "/content/drive/MyDrive/texture_frames/trigger"  # or "outputs/trigger"
model, tokenizer = train_trigger.train(
    base_model="microsoft/deberta-v3-large",
    output_dir=OUTPUT_DIR,
    epochs=5,
    batch_size=16,
    lr=1e-5,
    max_length=320,
)

## 5. Evaluate — word-level trigger F1 (baseline-comparable) + speed

In [ ]:
from eval_trigger import evaluate_trigger, print_report

metrics = evaluate_trigger(model, tokenizer, split="test", max_length=320)
print_report(metrics, reported_f1=0.735)

## Interpreting

This is the **scaffold** run — the goal is a working end-to-end pipeline and a
first honest F1, not necessarily beating 0.735 on attempt one. Two numbers to
report back:
1. **trigger F1** vs. 0.735, and
2. **ms/sentence** vs. the baseline's 196.6 ms/sample.

If F1 is in the ballpark (say >0.68) the pipeline is sound and Milestone 3 tuning
takes it the rest of the way. If it's near zero, it's almost always a
label-alignment bug — the Cell 3 sanity check is where we'd catch it.